# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors (FAIR²) Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and perform basic analysis on a research dataset described using a [Croissant schema](https://mlcommons.org/croissant/) and the `mlcroissant` Python library.

### Dataset Source
The dataset metadata and structure are defined at this URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

We will use the `mlcroissant` library to automatically discover and work with the described tabular data.

In [ ]:
# Install mlcroissant if needed. Uncomment below if not already installed.
!pip install mlcroissant

## 1. Data Loading
We begin by loading the Croissant schema and inspecting the metadata for the dataset.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# The published Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"\033[1mDataset name:\033[0m {metadata.name}")
print(f"\033[1mDescription:\033[0m {metadata.description}")
print(f"\033[1mIdentifier:\033[0m {getattr(metadata, 'identifier', '(none)')}")
print(f"\033[1mAvailable record sets:\033[0m", [rs.id for rs in getattr(metadata, 'record_set', [])])

## 2. Data Overview
Let us review the available record sets and their fields, each referenced by their `@id`.

We'll print the structure and the first record for each record set.

In [ ]:
# List all record sets and their fields (referenced by @id)
record_sets = getattr(metadata, 'record_set', [])
if not record_sets:
    # If Croissant-style key not set, infer from dataset (common in some schemas)
    try:
        # mlcroissant may provide fallback
        record_sets = dataset.record_sets()
    except Exception:
        record_sets = []

ids = []
for rs in record_sets:
    print(f'\nRecord Set: {rs.id}')
    ids.append(rs.id)
    try:
        fields = getattr(rs, 'field', [])
        if not fields:
            print('  (No fields listed)')
        for f in fields:
            print(f'  Field: {f.id} (type: {getattr(f, "data_type", "unknown")})')
    except Exception:
        print('  (Could not access field list)')
    # Print a sample record
    try:
        sample = next(dataset.records(record_set=rs.id))
        print(f'  Example record: {json.dumps(sample, indent=2)[:350]}...')
    except Exception as e:
        print(f'  No sample record could be loaded ({str(e)})')

## 3. Data Extraction
Extract records from a particular record set into a DataFrame for analysis.

Fill the following variable list with your selected record set `@id`s from the overview.

In [ ]:
# Replace with discovered record set IDs -- for this dataset, we check what exists
record_set_ids = [rs.id for rs in getattr(metadata, 'record_set', [])]

# To support fallback if above is missing (useful for partially described Croissant packages):
if not record_set_ids:
    try:
        record_set_ids = [rs.id for rs in dataset.record_sets()]
    except Exception:
        record_set_ids = []
if not record_set_ids:
    # Try common default (as often used in Croissant v1 schemas for a single table)
    record_set_ids = ['main']

dataframes = {}
for rs_id in record_set_ids:
    print(f'Loading records for record set: {rs_id}')
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f'\u2714 Loaded {len(df)} records for {rs_id} with columns: {df.columns.tolist()}')
    except Exception as e:
        print(f'Could not load records for {rs_id}:', str(e))

# For reference, print the columns of the (first) record set loaded
if dataframes:
    main_rs = record_set_ids[0]
    print(f'Columns for record set {main_rs}:')
    print(dataframes[main_rs].columns.tolist())
    dataframes[main_rs].head()

## 4. Exploratory Data Analysis (EDA)
Let's process and analyze some fields. We will:
- Select a numeric field (e.g. Age)
- Filter out records based on value
- Normalize the numeric field
- Group by a categorical field (e.g. Sex) and compute means
All variables will be referenced by their Croissant `@id`.

Update the variables below as needed to match field IDs discovered in Section 2/3.

In [ ]:
# Select record set and field ids (edit as appropriate for your dataset)
if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    # Try to guess likely numeric field IDs
    cand_numeric = [c for c in df.columns if 'age' in c.lower() or 'years' in c.lower()]
    # If known, can set e.g.: numeric_field_id = '@age_field_id_from_schema'
    numeric_field = cand_numeric[0] if cand_numeric else df.columns[0]
    print(f'Using numeric field: {numeric_field}')
    
    # Filter records: age > 50 (adjust as needed)
    threshold = 50
    try:
        filtered_df = df[df[numeric_field] > threshold]
    except Exception:
        # If non-numeric, try to convert
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        filtered_df = df[df[numeric_field] > threshold]
    print(f'Filtered records with {numeric_field} > {threshold}:')
    print(filtered_df.head())

    # Normalize
    filtered_df[f'{numeric_field}_normalized'] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
        filtered_df[numeric_field].std())
    print(f'Normalized {numeric_field} for filtered records:')
    print(filtered_df[[numeric_field, f'{numeric_field}_normalized']].head())

    # Pick a likely categorical field for grouping (e.g. sex or anatomical site)
    cand_group = [c for c in df.columns if 'sex' in c.lower() or 'site' in c.lower() or 'anatomical' in c.lower()]
    group_field = cand_group[0] if cand_group else None
    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f'Grouped by {group_field} (mean {numeric_field}):')
        print(grouped_df)
else:
    print('No DataFrame loaded.')

## 5. Visualization
Visualizing the distribution of the selected numeric field and its normalized variant.

We'll use matplotlib and seaborn to plot histograms of the numeric field before and after normalization, and boxplots by group if applicable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    if numeric_field in df.columns:
        plt.figure(figsize=(10,4))
        plt.subplot(1,2,1)
        sns.histplot(df[numeric_field].dropna(), bins=15, kde=True)
        plt.title(f'Histogram of {numeric_field}')

        if f'{numeric_field}_normalized' in df.columns:
            norm_vals = df[f'{numeric_field}_normalized'].dropna()
        else:
            norm_vals = (df[numeric_field] - df[numeric_field].mean()) / df[numeric_field].std()

        plt.subplot(1,2,2)
        sns.histplot(norm_vals, bins=15, kde=True)
        plt.title(f'Normalized {numeric_field}')
        plt.tight_layout()
        plt.show()

        # Boxplot by group field (if available)
        if group_field and group_field in df.columns:
            plt.figure(figsize=(7,4))
            sns.boxplot(x=group_field, y=numeric_field, data=df)
            plt.title(f'{numeric_field} by {group_field}')
            plt.show()
else:
    print('No data for visualization.')

## 6. Conclusion

In this notebook, we used `mlcroissant` to load the FAIR² clinical oncology dataset, explored schema and records using entity `@id`s, and performed basic processing and visualization on tabular data. This approach supports reproducibility, data documentation, and efficient analysis directly from standardized, machine-readable metadata.